In [2]:
import logging
from pathlib import Path

import numpy as np
import xarray as xr

In [3]:
# Setup basic logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
def extract_and_combine_wrf_vars(download_path, file_2d_1, file_2d_2, file_3d_1, file_3d_2):
    """
    Extract and combine specific variables from WRF 2D and 3D files

    Args:
        download_path: Base path where files are located
        file_2d_1: First 2D filename
        file_2d_2: Second 2D filename
        file_3d_1: First 3D filename
        file_3d_2: Second 3D filename

    Returns:
        Dictionary containing:
        - atmos_vars: 3D atmospheric variables (z, t, u, v, q)
        - surf_vars: Surface variables (t2, u10, v10, psfc)
        - static_vars: Static variables (z)
        - metadata: Metadata (pressure levels, lat, lon, time)
    """
    try:
        base_path = Path(download_path)

        # Load 2D files
        logger.info("Loading 2D files...")
        ds_2d_1 = xr.open_dataset(base_path / file_2d_1, engine="netcdf4", decode_times=False)
        ds_2d_2 = xr.open_dataset(base_path / file_2d_2, engine="netcdf4", decode_times=False)

        # Extract metadata and surface variables from 2D files
        logger.info("Extracting metadata and 2D variables...")
        metadata = {
            "lat": np.asarray(ds_2d_1["XLAT"].values[0], dtype=np.float32),
            "lon": np.asarray(ds_2d_1["XLONG"].values[0], dtype=np.float32),
            "time": np.asarray([ds_2d_1["Times"].values[0], ds_2d_2["Times"].values[0]]),
        }

        surf_vars = {
            "t2": np.asarray(
                np.concatenate([ds_2d_1["T2"].values, ds_2d_2["T2"].values], axis=0),
                dtype=np.float32,
            ),
            "u10": np.asarray(
                np.concatenate([ds_2d_1["U10"].values, ds_2d_2["U10"].values], axis=0),
                dtype=np.float32,
            ),
            "v10": np.asarray(
                np.concatenate([ds_2d_1["V10"].values, ds_2d_2["V10"].values], axis=0),
                dtype=np.float32,
            ),
            "psfc": np.asarray(
                np.concatenate([ds_2d_1["PSFC"].values, ds_2d_2["PSFC"].values], axis=0),
                dtype=np.float32,
            ),
        }

        static_vars = {"z": np.asarray(ds_2d_1["Z"].values[0], dtype=np.float32)}

        # Close 2D datasets
        ds_2d_1.close()
        ds_2d_2.close()
        del ds_2d_1, ds_2d_2

        # Load 3D files
        logger.info("Loading 3D files...")
        ds_3d_1 = xr.open_dataset(base_path / file_3d_1, engine="netcdf4", decode_times=False)
        ds_3d_2 = xr.open_dataset(base_path / file_3d_2, engine="netcdf4", decode_times=False)

        # Extract pressure levels and atmospheric variables
        metadata["pressure_levels"] = np.asarray(ds_3d_1["P"].values[0], dtype=np.float32)

        atmos_vars = {
            "z": np.asarray(
                np.concatenate([ds_3d_1["Z"].values, ds_3d_2["Z"].values], axis=0), dtype=np.float32
            ),
            "t": np.asarray(
                np.concatenate([ds_3d_1["TK"].values, ds_3d_2["TK"].values], axis=0),
                dtype=np.float32,
            ),
            "u": np.asarray(
                np.concatenate([ds_3d_1["U"].values, ds_3d_2["U"].values], axis=0), dtype=np.float32
            ),
            "v": np.asarray(
                np.concatenate([ds_3d_1["V"].values, ds_3d_2["V"].values], axis=0), dtype=np.float32
            ),
            "q": np.asarray(
                np.concatenate([ds_3d_1["QVAPOR"].values, ds_3d_2["QVAPOR"].values], axis=0),
                dtype=np.float32,
            ),
        }

        # Close 3D datasets
        ds_3d_1.close()
        ds_3d_2.close()
        del ds_3d_1, ds_3d_2

        # Log shapes and dtypes for verification
        logger.info("\nMetadata Shapes and Dtypes:")
        for var_name, data in metadata.items():
            logger.info(f"{var_name}: type={type(data)}, shape={data.shape}, dtype={data.dtype}")

        logger.info("\nSurface Variables Shapes and Dtypes:")
        for var_name, data in surf_vars.items():
            logger.info(f"{var_name}: type={type(data)}, shape={data.shape}, dtype={data.dtype}")

        logger.info("\nStatic Variables Shapes and Dtypes:")
        for var_name, data in static_vars.items():
            logger.info(f"{var_name}: type={type(data)}, shape={data.shape}, dtype={data.dtype}")

        logger.info("\nAtmospheric Variables Shapes and Dtypes:")
        for var_name, data in atmos_vars.items():
            logger.info(f"{var_name}: type={type(data)}, shape={data.shape}, dtype={data.dtype}")

        return {
            "atmos_vars": atmos_vars,
            "surf_vars": surf_vars,
            "static_vars": static_vars,
            "metadata": metadata,
        }

    except Exception as e:
        logger.error(f"Error processing files: {e}")
        raise

In [4]:
# Example usage:
if __name__ == "__main__":
    # Replace these with your actual paths and filenames
    download_path = "/home/user/Documents/aurora/data_wrf"
    file_2d_1 = "2d/wrf2d_d01_2015-12-01_00:00:00.nc"
    file_2d_2 = "2d/wrf2d_d01_2015-12-01_03:00:00.nc"
    file_3d_1 = "3d/wrf3d_d01_2015-12-01_00:00:00.nc"
    file_3d_2 = "3d/wrf3d_d01_2015-12-01_03:00:00.nc"

    data = extract_and_combine_wrf_vars(download_path, file_2d_1, file_2d_2, file_3d_1, file_3d_2)

    # Access the combined variables
    static_vars = data["static_vars"]
    surf_vars = data["surf_vars"]
    atmos_vars = data["atmos_vars"]
    metadata = data["metadata"]

INFO:__main__:Loading 2D files...
INFO:__main__:Loading 3D files...
INFO:__main__:Extracting metadata...
INFO:__main__:Extracting 2D variables...
INFO:__main__:Extracting static variables...
INFO:__main__:Extracting 3D variables...
INFO:__main__:
Metadata Shapes and Dtypes:
INFO:__main__:pressure_levels: type=<class 'numpy.ndarray'>, shape=(50, 1419, 1429), dtype=float32
INFO:__main__:lat: type=<class 'numpy.ndarray'>, shape=(1429,), dtype=float32
INFO:__main__:lon: type=<class 'numpy.ndarray'>, shape=(1429,), dtype=float32
INFO:__main__:time: type=<class 'numpy.ndarray'>, shape=(2,), dtype=|S19
INFO:__main__:
Surface Variables Shapes and Dtypes:
INFO:__main__:t2: type=<class 'numpy.ndarray'>, shape=(2, 1419, 1429), dtype=float32
INFO:__main__:u10: type=<class 'numpy.ndarray'>, shape=(2, 1419, 1429), dtype=float32
INFO:__main__:v10: type=<class 'numpy.ndarray'>, shape=(2, 1419, 1429), dtype=float32
INFO:__main__:psfc: type=<class 'numpy.ndarray'>, shape=(2, 1419, 1429), dtype=float32
I

: 

In [7]:
def standardize_3d_shapes(atmos_vars):
    """
    Standardize the shapes of 3D atmospheric variables to (2, 50, 1419, 1429)

    Args:
        atmos_vars: Dictionary containing 3D atmospheric variables

    Returns:
        Dictionary with standardized shapes for all variables
    """
    try:
        logger.info("Standardizing 3D variable shapes...")

        # Create a copy to avoid modifying the original
        standardized_vars = {}

        for var_name, data in atmos_vars.items():
            # Handle z variable (51 levels)
            if var_name == "z" and data.shape[1] == 51:
                # Take only the first 50 levels
                standardized_vars[var_name] = data[:, :50, :, :]
                logger.info("Reduced z levels from 51 to 50")

            # Handle u variable (1430 columns)
            elif var_name == "u" and data.shape[3] == 1430:
                # Take only the first 1429 columns
                standardized_vars[var_name] = data[:, :, :, :1429]
                logger.info("Reduced u columns from 1430 to 1429")

            # Handle v variable (1420 rows)
            elif var_name == "v" and data.shape[2] == 1420:
                # Take only the first 1419 rows
                standardized_vars[var_name] = data[:, :, :1419, :]
                logger.info("Reduced v rows from 1420 to 1419")

            # Variables that already have correct shape
            else:
                standardized_vars[var_name] = data

        # Log the new shapes
        logger.info("\nStandardized 3D Variables Shapes:")
        for var_name, data in standardized_vars.items():
            logger.info(f"{var_name}: {data.shape}")

        return standardized_vars

    except Exception as e:
        logger.error(f"Error standardizing shapes: {e}")
        raise

In [8]:
standardized_atmos_vars = standardize_3d_shapes(atmos_vars)

INFO:__main__:Standardizing 3D variable shapes...
INFO:__main__:Reduced z levels from 51 to 50
INFO:__main__:Reduced u columns from 1430 to 1429
INFO:__main__:Reduced v rows from 1420 to 1419
INFO:__main__:
Standardized 3D Variables Shapes:
INFO:__main__:z: (2, 50, 1419, 1429)
INFO:__main__:t: (2, 50, 1419, 1429)
INFO:__main__:u: (2, 50, 1419, 1429)
INFO:__main__:v: (2, 50, 1419, 1429)
INFO:__main__:q: (2, 50, 1419, 1429)


In [9]:
def transform_metadata_coordinates(metadata):
    """
    Transform coordinates in metadata to match Aurora's requirements:
    - Latitude: -90 to 90 degrees, in decreasing order
    - Longitude: 0 to 360 degrees

    Args:
        metadata: Dictionary containing lat and lon arrays

    Returns:
        Dictionary with transformed coordinates
    """
    try:
        logger.info("Transforming coordinates in metadata...")

        # Create a copy to avoid modifying the original
        transformed_metadata = metadata.copy()

        # Check and transform latitude
        lat = metadata["lat"]
        lat_min, lat_max = np.min(lat), np.max(lat)
        logger.info(f"Original latitude range: {lat_min} to {lat_max}")

        if lat_min < -90 or lat_max > 90:
            logger.warning("Latitude values outside -90 to 90 range detected")
            # Clip values to valid range
            transformed_metadata["lat"] = np.clip(lat, -90, 90)
            logger.info(
                f"Latitude clipped to range: {np.min(transformed_metadata['lat'])} to {np.max(transformed_metadata['lat'])}"
            )

        # Sort latitude in decreasing order
        transformed_metadata["lat"] = np.sort(transformed_metadata["lat"])[::-1]
        logger.info("Latitude sorted in decreasing order")

        # Check and transform longitude
        lon = metadata["lon"]
        lon_min, lon_max = np.min(lon), np.max(lon)
        logger.info(f"Original longitude range: {lon_min} to {lon_max}")

        if lon_min < 0 or lon_max > 360:
            logger.info("Converting longitude from -180/180 to 0/360 range")
            # Convert -180/180 to 0/360 using (lon + 360) % 360
            transformed_metadata["lon"] = (lon + 360) % 360
            logger.info(
                f"Longitude transformed to range: {np.min(transformed_metadata['lon'])} to {np.max(transformed_metadata['lon'])}"
            )

        # Log final ranges
        logger.info("\nFinal coordinate ranges:")
        logger.info(
            f"Latitude: {np.min(transformed_metadata['lat'])} to {np.max(transformed_metadata['lat'])}"
        )
        logger.info(
            f"Longitude: {np.min(transformed_metadata['lon'])} to {np.max(transformed_metadata['lon'])}"
        )

        return transformed_metadata

    except Exception as e:
        logger.error(f"Error transforming coordinates in metadata: {e}")
        raise

In [10]:
metadata_transformed = transform_metadata_coordinates(metadata=metadata)

INFO:__main__:Transforming coordinates in metadata...
INFO:__main__:Original latitude range: 15.028518676757812 to 22.810646057128906
INFO:__main__:Latitude sorted in decreasing order
INFO:__main__:Original longitude range: -130.53163146972656 to -81.0020751953125
INFO:__main__:Converting longitude from -180/180 to 0/360 range
INFO:__main__:Longitude transformed to range: 229.46836853027344 to 278.9979248046875
INFO:__main__:
Final coordinate ranges:
INFO:__main__:Latitude: 15.028518676757812 to 22.810646057128906
INFO:__main__:Longitude: 229.46836853027344 to 278.9979248046875


In [11]:
def process_pressure_levels(pressure_levels):
    """
    Process pressure levels to create 50 distinct integer pressure levels.

    Args:
        pressure_levels: Array of pressure levels from the dataset

    Returns:
        Array of 50 distinct integer pressure levels
    """
    try:
        logger.info("Processing pressure levels...")

        # Convert pressure levels to numpy array if it's not already
        p_levels = np.array(pressure_levels)

        # Get min and max pressure levels
        p_min, p_max = np.min(p_levels), np.max(p_levels)
        logger.info(f"Original pressure range: {p_min} to {p_max} hPa")

        # Create 50 evenly spaced integer pressure levels between min and max
        processed_levels = np.linspace(p_min, p_max, 50)
        # Round to nearest integer
        processed_levels = np.round(processed_levels).astype(int)
        # Ensure all levels are unique
        processed_levels = np.unique(processed_levels)

        # If we have fewer than 50 unique levels, add more by incrementing
        if len(processed_levels) < 50:
            additional_levels = np.arange(
                processed_levels[-1] + 1, processed_levels[-1] + (50 - len(processed_levels)) + 1
            )
            processed_levels = np.concatenate([processed_levels, additional_levels])

        logger.info(
            f"Created {len(processed_levels)} distinct integer pressure levels from {processed_levels[0]} to {processed_levels[-1]} hPa"
        )

        return processed_levels

    except Exception as e:
        logger.error(f"Error processing pressure levels: {e}")
        raise

In [12]:
processed_p_levels = process_pressure_levels(metadata_transformed["pressure_levels"])
metadata_transformed["pressure_levels"] = processed_p_levels

INFO:__main__:Processing pressure levels...
INFO:__main__:Original pressure range: 5141.2021484375 to 105285.96875 hPa
INFO:__main__:Created 50 distinct integer pressure levels from 5141 to 105286 hPa


In [13]:
def log_variable_shapes(data_dict, dict_name):
    """
    Log the shapes and types of variables in a dictionary.

    Args:
        data_dict: Dictionary containing variables
        dict_name: Name of the dictionary for logging
    """
    logger.info(f"\nShapes and types of variables in {dict_name}:")
    for var_name, data in data_dict.items():
        logger.info(f"{var_name}:")
        logger.info(
            f"  Shape: {data.shape} :  Type: {type(data)} :  Dtype: {data.dtype if hasattr(data, 'dtype') else 'N/A'}: Is numpy: {isinstance(data, np.ndarray)} "
        )


# Check shapes and types of all dictionaries
log_variable_shapes(static_vars, "static_vars")
log_variable_shapes(surf_vars, "surf_vars")
log_variable_shapes(standardized_atmos_vars, "standardized_atmos_vars")
log_variable_shapes(metadata_transformed, "metadata_transformed")

INFO:__main__:
Shapes and types of variables in static_vars:
INFO:__main__:z:
INFO:__main__:  Shape: (1419, 1429) :  Type: <class 'numpy.ndarray'> :  Dtype: float32: Is numpy: True 
INFO:__main__:
Shapes and types of variables in surf_vars:
INFO:__main__:t2:
INFO:__main__:  Shape: (2, 1419, 1429) :  Type: <class 'numpy.ndarray'> :  Dtype: float32: Is numpy: True 
INFO:__main__:u10:
INFO:__main__:  Shape: (2, 1419, 1429) :  Type: <class 'numpy.ndarray'> :  Dtype: float32: Is numpy: True 
INFO:__main__:v10:
INFO:__main__:  Shape: (2, 1419, 1429) :  Type: <class 'numpy.ndarray'> :  Dtype: float32: Is numpy: True 
INFO:__main__:psfc:
INFO:__main__:  Shape: (2, 1419, 1429) :  Type: <class 'numpy.ndarray'> :  Dtype: float32: Is numpy: True 
INFO:__main__:
Shapes and types of variables in standardized_atmos_vars:
INFO:__main__:z:
INFO:__main__:  Shape: (2, 50, 1419, 1429) :  Type: <class 'numpy.ndarray'> :  Dtype: float32: Is numpy: True 
INFO:__main__:t:
INFO:__main__:  Shape: (2, 50, 1419, 

In [14]:
import torch

from aurora import Batch, Metadata

/home/user/anaconda3/envs/aurora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [82]:
# Create tensors first
surf_tensors = {
    "2t": torch.tensor(surf_vars["t2"], dtype=torch.float32)[None],
    "10u": torch.tensor(surf_vars["u10"], dtype=torch.float32)[None],
    "10v": torch.tensor(surf_vars["v10"], dtype=torch.float32)[None],
    "msl": torch.tensor(surf_vars["psfc"], dtype=torch.float32)[None],
}

static_tensors = {
    "z": torch.tensor(static_vars["z"], dtype=torch.float32),
}

atmos_tensors = {
    "t": torch.tensor(standardized_atmos_vars["t"], dtype=torch.float32)[None],
    "u": torch.tensor(standardized_atmos_vars["u"], dtype=torch.float32)[None],
    "v": torch.tensor(standardized_atmos_vars["v"], dtype=torch.float32)[None],
    "q": torch.tensor(standardized_atmos_vars["q"], dtype=torch.float32)[None],
    "z": torch.tensor(standardized_atmos_vars["z"], dtype=torch.float32)[None],
}

# Convert time strings to datetime objects
time_strings = metadata_transformed["time"]
# Convert to datetime64 and then to datetime objects
time_objects = tuple(
    np.datetime64(t.decode("utf-8").replace("_", " ")).astype("datetime64[s]").tolist()
    for t in time_strings
)

metadata_tensors = Metadata(
    lat=torch.tensor(metadata_transformed["lat"], dtype=torch.float32),
    lon=torch.tensor(metadata_transformed["lon"], dtype=torch.float32),
    time=time_objects,
    atmos_levels=tuple(metadata_transformed["pressure_levels"]),
)

# Create the batch with the tensors
batch = Batch(
    surf_vars=surf_tensors,
    static_vars=static_tensors,
    atmos_vars=atmos_tensors,
    metadata=metadata_tensors,
)

: 